# R3-M1: BiGRU under Protocol B
Select a Colab T4 runtime. Sync the updated project code, tests, frozen manifests, R2-S1 files and cohort shards to Drive. Run cells in order. Training, checkpoints and the ledger are written to Drive. No test or Chatterbox audio is scored.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import csv, hashlib, json, os, subprocess, sys, time
ROOT = Path('/content/drive/MyDrive/PAI&AIES Project')  # Change if your project is elsewhere.
SHARDS = ROOT / 'data/processed/cohort_v1'  # Or point to a verified local scratch copy.
LEDGER = ROOT / 'reports/ledger.csv'
# For a new run leave RUN_ID blank. To resume, enter the exact previous RUN_ID.
RUN_ID = ''
RESUME = bool(RUN_ID)
if not RESUME:
    RUN_ID = 'r3_m1_' + time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())
OUTPUT = ROOT / 'runs/r3' / RUN_ID
print('Run:', RUN_ID, 'Resume:', RESUME, 'Output:', OUTPUT)
assert (ROOT / 'scripts/r3_m1_train.py').is_file(), 'Sync the updated project code to Drive'
assert SHARDS.is_dir(), 'Set SHARDS to the canonical shard directory'
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')


In [ ]:
import torch
assert torch.cuda.is_available(), 'Choose a GPU runtime in Colab'
print('GPU:', torch.cuda.get_device_name(0), 'PyTorch:', torch.__version__)
required = ['data/manifests/cohort_v1.parquet', 'data/splits/protocol_B_v1.parquet',
            'data/processed/chunks_v1.parquet', 'data/features/r2_s1_B/r2_s1_manifest.json',
            'data/features/r2_s1_B/mel_stats_B_v1.npz', 'reports/r1/split_report.json']
missing = [p for p in required if not (ROOT / p).is_file()]
assert not missing, f'Missing frozen inputs: {missing}'
assert (OUTPUT / 'experiment_contract.json').is_file() if RESUME else not OUTPUT.exists(), 'Check RUN_ID and resume setting'


## Fixture tests and GPU preprocessing parity
A failure or skipped frontend test must be resolved before training. The runner repeats the GPU frontend check.


In [ ]:
tests = ['tests/test_r3_m1_data.py', 'tests/test_r3_m1_selection.py', 'tests/test_r3_m1_export.py',
         'tests/test_r3_m1_orchestration.py', 'tests/test_r3_bigru_frontend.py', 'tests/test_m1_trainer.py']
check = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', *tests], text=True, capture_output=True)
print(check.stdout, check.stderr)
assert check.returncode == 0 and ' skipped' not in check.stdout, 'Resolve test failures or skips before training'


## Train and export
Do not run another process that writes to the same ledger at the same time. If disconnected, reconnect with the same code, environment and RUN_ID; enter it above and rerun the notebook. Exact environment and hash matching is required for resume.


In [ ]:
cmd = [sys.executable, 'scripts/r3_m1_train.py', '--root', str(ROOT), '--shard-dir', str(SHARDS),
       '--output', str(OUTPUT), '--ledger', str(LEDGER)]
if RESUME:
    cmd.append('--resume')
subprocess.run(cmd, check=True)


## Verify handoff and write report
The report verifies recorded file hashes and completion rows; it does not substitute for independent review or final evaluation.


In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1048576), b''):
            h.update(block)
    return h.hexdigest()
evidence = json.loads((OUTPUT / 'evidence.json').read_text(encoding='utf-8'))
assert evidence['status'] == 'validation_predictions_complete'
assert sha256(OUTPUT / 'selection.json') == evidence['selection_sha256']
assert all(sha256(OUTPUT / name) == digest for name, digest in evidence['output_sha256'].items())
with LEDGER.open(newline='', encoding='utf-8') as f:
    ledger_rows = list(csv.DictReader(f))
completed = [r for r in ledger_rows if r['run_id'] in {RUN_ID + '-baseline', RUN_ID + '-lower_lr'} and r['status'] == 'completed']
assert {r['run_id'] for r in completed} == {RUN_ID + '-baseline', RUN_ID + '-lower_lr'}
for name, digest in evidence['selection']['checkpoint_sha256'].items():
    assert sha256(OUTPUT / (RUN_ID + '-' + name) / 'best.pt') == digest
report = {'task': 'R3-M1', 'run_id': RUN_ID, 'output': str(OUTPUT), 'evidence_sha256': sha256(OUTPUT / 'evidence.json'),
          'evidence': evidence, 'completed_ledger_rows': completed, 'gpu': torch.cuda.get_device_name(0),
          'verification': 'artifact hashes and completed ledger rows verified; independent review pending'}
report_path = ROOT / 'reports/r3' / (RUN_ID + '.json')
report_path.parent.mkdir(parents=True, exist_ok=True)
if report_path.exists():
    raise FileExistsError(f'Report already exists: {report_path}')
with report_path.open('x', encoding='utf-8') as f:
    json.dump(report, f, indent=2, allow_nan=False)
print('Share report:', report_path, '\nRun artifacts:', OUTPUT, '\nLedger:', LEDGER)
